## Segmentation

The intent is a three-stage post-training pipeline:

Load a trained model, run inference on a validation image, and visualise the predicted segmentation mask overlaid on the original satellite image alongside ground truth.

Take the predicted mask, resize it to 224×224, then convert it into a coarse block-level ROI map (16×16 blocks). This identifies which regions of the image contain "important" land-use classes vs background.

Use that ROI map to apply different processing to important vs unimportant regions: bilateral filter smoothing on non-ROI areas, and the intended (but missing) JPEG-XL compression step.

Stage 1 - broken, two issues:


image = valid_dataset[0]['original_image']   # key doesn't exist
...
valid_dataset[0]['original_map']             # key doesn't exist
The dataset class only returns pixel_values and labels. There's no original_image or original_map key. This will crash immediately.

Cells 20-22 — would work correctly if cell-18 ran successfully, since they only depend on predicted_segmentation_map which cell-18 produces.

Cell-23 — broken:


new_img = jpegxl_decode(compressed)
compressed is never defined anywhere. jpegxl_decode is never imported. This is an unfinished stub.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def ade_palette():
    return [
        [0, 255, 255],
        [255, 255, 0],
        [255, 0, 255],
        [0, 255, 0],
        [0, 0, 255],
        [255, 255, 255],
        [0, 0, 0]
    ]

image = valid_dataset[0]['original_image']

pixel_values = image_processor(image.permute(1, 2, 0).numpy(), return_tensors="pt").pixel_values.to("cuda")

model.eval()
with torch.no_grad():
    outputs = model(pixel_values=pixel_values)

predicted_segmentation_map = image_processor.post_process_semantic_segmentation(outputs, target_sizes=[[image.shape[1], image.shape[2]]])[0]
predicted_segmentation_map = predicted_segmentation_map.cpu()

color_seg = np.zeros((predicted_segmentation_map.shape[0], predicted_segmentation_map.shape[1], 3), dtype=np.uint8)

palette = np.array(ade_palette())
for label, color in enumerate(palette):
    color_seg[predicted_segmentation_map == label, :] = color

color_seg = color_seg[..., ::-1]

img = np.moveaxis(image.numpy(), 0, -1) * 0.5 + color_seg * 0.5

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(np.moveaxis(image.numpy(), 0, -1).astype(np.uint8))
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img.astype(np.uint8))
plt.title('Prediction Overlay')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(valid_dataset[0]['original_map'].numpy())
plt.title('Ground Truth')
plt.axis('off')

plt.tight_layout()
plt.show()

print("=== Compression Section ===")
print("This section demonstrates ROI-based compression using your segmentation results.")
print("You can apply different compression strategies to ROI vs non-ROI regions.")

In [ ]:
img_mv = np.moveaxis(img,-1,0)
print(predicted_segmentation_map.shape)
resized_mask = F.interpolate(predicted_segmentation_map.unsqueeze(0).unsqueeze(0).to(torch.uint8), size = (224,224), mode='nearest').squeeze(0).squeeze(0)
print(resized_mask.shape)
resized_mask_np = resized_mask.numpy()





torch.Size([512, 512])
torch.Size([224, 224])


In [ ]:
from skimage.measure import block_reduce

print("Creating ROI blocks from segmentation mask...")

resized_mask = torch.nn.functional.interpolate(
    predicted_segmentation_map.unsqueeze(0).unsqueeze(0).float(),
    size=(224, 224),
    mode='nearest'
).squeeze(0).squeeze(0)

resized_mask_np = resized_mask.numpy()

block_size = (16, 16)
roi_block = block_reduce(resized_mask_np, block_size=block_size, func=np.mean)
roi_block = np.where(roi_block > 0.3, 1, 0)

roi_coords = []
for i in range(roi_block.shape[0]):
    for j in range(roi_block.shape[1]):
        if roi_block[i, j] == 1:
            x_start = i * block_size[0]
            y_start = j * block_size[1]
            x_end = x_start + block_size[0]
            y_end = y_start + block_size[1]
            roi_coords.append((x_start, y_start, x_end, y_end))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(resized_mask_np, cmap='gray')
axes[0].set_title('Segmentation Mask (Pixel-level)')
axes[1].imshow(roi_block, cmap='gray', interpolation='nearest')
axes[1].set_title('Block-level ROI (16x16 blocks)')
plt.tight_layout()
plt.show()

print(f"Found {len(roi_coords)} ROI blocks")

In [ ]:
import cv2

def apply_bilateral_filter(image, mask, d=9, sigma_color=75, sigma_space=75):
    result = image.copy()
    if len(image.shape) == 3:
        for i in range(3):
            channel = image[:, :, i]
            filtered_channel = cv2.bilateralFilter(channel.astype(np.uint8), d, sigma_color, sigma_space)
            result[:, :, i] = np.where(mask[:, :, i], filtered_channel, channel)
    else:
        filtered = cv2.bilateralFilter(image.astype(np.uint8), d, sigma_color, sigma_space)
        result = np.where(mask, filtered, image)
    return result

print("Applying bilateral filter to non-ROI regions...")

original_img = valid_dataset[0]['original_image'].permute(1, 2, 0).numpy()

mask_2d = (resized_mask_np > 0).astype(bool)
mask_3d = np.repeat(mask_2d[:, :, np.newaxis], 3, axis=2)
inverse_mask_3d = ~mask_3d

resized_original = cv2.resize(original_img, (224, 224))

filtered_img = apply_bilateral_filter(resized_original, inverse_mask_3d, d=9, sigma_color=75, sigma_space=75)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(resized_original.astype(np.uint8))
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(filtered_img.astype(np.uint8))
axes[1].set_title('Bilateral Filtered (non-ROI)')
axes[1].axis('off')

diff = np.abs(resized_original.astype(float) - filtered_img.astype(float)).mean(axis=2)
axes[2].imshow(diff, cmap='hot')
axes[2].set_title('Difference Map')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("Bilateral filter applied to preserve edges while smoothing non-ROI regions")

In [ ]:
new_img = jpegxl_decode(compressed)

In [ ]:
print("Note: Synthesis/Extrapolation section requires Sentinel-2 .npz data format.")
print("Current dataset uses RGB .jpg images which don't have SWIR bands.")
print("This section is skipped for the archive dataset.")